# Downloading the data

Please download the original data from these two Zenodo links:
1. https://zenodo.org/records/15394341
2. https://zenodo.org/records/15388782
If you intend to use this data, remember to credit the original owners of the data, mentioned in the other notebook.

# Flatten primary manufacturing JSON dataset

Nested lists (operations, conditions, materials) are
normalized into separate tables, linked by the IDs already present in the
source data (`procedure_id`, `op_id`).

Output: 5 CSV files, joinable via `procedure_id` / `op_id`.

Only the input paths needs editing.

In [1]:
import json
from pathlib import Path

import pandas as pd

In [ ]:
# --- EDIT THESE TWO PATHS ---
INPUT_DIR = Path(r"Add Path Here")
OUTPUT_DIR = Path(r"Add Path Here")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
def extract_procedure(data):

    pid = data.get("procedure_id")

    procedure_row = {
        "procedure_id": pid,
        "patent_number": data.get("patent_number"),
        "title": data.get("title"),
        "target_name": data.get("target_name"),
        "target_InChI": data.get("target_InChI"),
        "target_InChIKey": data.get("target_InChIKey"),
        "yield_amount": json.dumps(data.get("yield_amount")),
        "yield_percentage": json.dumps(data.get("yield_percentage")),
    }

    operation_rows = []
    condition_rows = []
    op_material_rows = []

    for op in (data.get("operations") or []):
        if not isinstance(op, dict):
            continue
        op_id = op.get("op_id")

        operation_rows.append({
            "procedure_id": pid,
            "op_id": op_id,
            "op_raw_data": op.get("op_raw_data"),
            "op_std_action": op.get("op_std_action"),
        })

        for cond in (op.get("op_conditions") or []):
            if not isinstance(cond, dict):
                continue
            condition_rows.append({
                "op_id": op_id,
                "cond_id": cond.get("cond_id"),
                "cond_raw_data": cond.get("cond_raw_data"),
                "cond_parameter": cond.get("cond_parameter"),
                "cond_value": json.dumps(cond.get("cond_value")),
                "cond_units": cond.get("cond_units"),
            })

        for mat in (op.get("op_materials") or []):
            if not isinstance(mat, dict):
                continue
            op_material_rows.append({
                "op_id": op_id,
                "mat_id": mat.get("mat_id"),
                "mat_name": mat.get("mat_name"),
                "mat_type": mat.get("mat_type"),
                "mat_InChI": mat.get("mat_InChI"),
                "mat_InChIKey": mat.get("mat_InChIKey"),
                "mat_amount": mat.get("mat_amount"),
                "mat_concentration": mat.get("mat_concentration"),
                "mat_category": mat.get("mat_category"),
                "mat_ref": mat.get("mat_ref"),
                "mat_status": mat.get("mat_status"),
            })

    materials_reference_rows = []
    for mat in (data.get("materials") or []):
        if not isinstance(mat, dict):
            continue
        materials_reference_rows.append({
            "procedure_id": pid,
            "mat_ref": mat.get("mat_ref"),
            "mat_name": mat.get("mat_name"),
            "mat_type": mat.get("mat_type"),
            "mat_InChI": mat.get("mat_InChI"),
            "mat_InChIKey": mat.get("mat_InChIKey"),
            "mat_status": mat.get("mat_status"),
        })

    return procedure_row, operation_rows, condition_rows, op_material_rows, materials_reference_rows

In [ ]:
json_files = sorted(INPUT_DIR.glob("*.json"))
total = len(json_files)
print(f"Found {total} JSON files in {INPUT_DIR}")

if total == 0:
    raise SystemExit("No .json files found — check INPUT_DIR.")

In [ ]:
procedures, operations, conditions, op_materials, materials_reference = [], [], [], [], []
failed = []

for i, fp in enumerate(json_files, 1):
    try:
        with open(fp, "r", encoding="utf-8") as f:
            data = json.load(f)
        p_row, op_rows, cond_rows, opmat_rows, matref_rows = extract_procedure(data)
        procedures.append(p_row)
        operations.extend(op_rows)
        conditions.extend(cond_rows)
        op_materials.extend(opmat_rows)
        materials_reference.extend(matref_rows)
    except Exception as e:
        failed.append(f"{fp.name}: {e}")

    if i % 10000 == 0 or i == total:
        print(f"  processed {i}/{total}")

print(f"Done. {len(procedures)} procedures parsed, {len(failed)} failed.")

In [ ]:
tables = {
    "procedures": pd.DataFrame(procedures),
    "operations": pd.DataFrame(operations),
    "op_conditions": pd.DataFrame(conditions),
    "op_materials": pd.DataFrame(op_materials),
    "materials_reference": pd.DataFrame(materials_reference),
}

for name, t in tables.items():
    t.to_csv(OUTPUT_DIR / f"{name}.csv", index=False)
    print(f"{name}: {t.shape[0]} rows, {t.shape[1]} columns -> {name}.csv")

if failed:
    with open(OUTPUT_DIR / "failed_files.txt", "w", encoding="utf-8") as f:
        f.write("\n".join(failed))
    print(f"{len(failed)} files failed to parse — see failed_files.txt")

print(f"\nOutput written to: {OUTPUT_DIR}")

## What's in each file

- **procedures.csv** — one row per procedure. `procedure_id`, `patent_number`, `title`, `target_name`, `target_InChI`, `target_InChIKey`, `yield_amount`, `yield_percentage` (last two are the original lists, kept as JSON text).
- **operations.csv** — one row per step. `procedure_id`, `op_id`, `op_raw_data`, `op_std_action`. Join to `procedures` on `procedure_id`.
- **op_conditions.csv** — one row per condition attached to a step. `op_id`, `cond_id`, `cond_raw_data`, `cond_parameter`, `cond_value` (original list/string, kept as JSON text), `cond_units`. Join to `operations` on `op_id`.
- **op_materials.csv** — one row per material used in a step. `op_id`, `mat_id`, `mat_name`, `mat_type`, `mat_InChI`, `mat_InChIKey`, `mat_amount`, `mat_concentration`, `mat_category`, `mat_ref`, `mat_status`. Join to `operations` on `op_id`.
- **materials_reference.csv** — the top-level deduped materials list per procedure. `procedure_id`, `mat_ref`, `mat_name`, `mat_type`, `mat_InChI`, `mat_InChIKey`, `mat_status`.